# DNABERT-2 베이스 모델 출력 확인과 DNA 서열 분류

이 노트북은 다음 두 단계로 실행됩니다.
1. 공식 `zhihan1996/DNABERT-2-117M` 체크포인트를 불러와 토큰 임베딩을 확인합니다.
2. 이진 분류 head를 추가하고 학습한 뒤 validation/test 성능 지표를 출력합니다.

기본값에서는 모든 셀의 동작을 확인할 수 있도록 작은 합성 motif 데이터셋을 사용합니다. 실제 실험에서는 `DATA_PATH`에 `sequence,label` 열이 있는 CSV 파일을 지정하세요. 합성 데이터의 결과는 파이프라인 작동 확인용이며 생물학적 성능 지표가 아닙니다.


## 0. 실행 환경 설정

DNABERT-2는 Hugging Face의 원격 모델 코드를 사용합니다. 이 프로젝트는 Python 3.11 전용 가상환경과 `Python 3.11 (DNABERT2 HE)` 커널로 설정되어 있습니다.

In [ ]:
import sys

print(sys.executable)

In [ ]:
# 이 노트북은 프로젝트 내부의 Python 3.11 가상환경에 맞춰 설정되어 있습니다.

import os
import random
import sys
from pathlib import Path

import truststore
truststore.inject_into_ssl()  # Hugging Face 다운로드에 Windows 인증서 저장소를 사용합니다.
os.environ.setdefault("HF_HOME", str(Path.cwd() / ".cache" / "huggingface"))
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score, average_precision_score, classification_report,
    confusion_matrix, f1_score, matthews_corrcoef, precision_score,
    recall_score, roc_auc_score,
)
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer
from transformers.models.bert.configuration_bert import BertConfig

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("실행 장치:", DEVICE)
if DEVICE.type == "cuda":
    print("사용 GPU:", torch.cuda.get_device_name(0))


## 1. 학습 설정

처음 작동을 확인할 때는 `FREEZE_ENCODER=True`를 유지합니다. 실제 전체 파인튜닝에서는 `False`로 바꾸고 학습률을 낮춘 뒤(예: `2e-5`) GPU로 학습하세요. 8GB VRAM에서는 전체 파인튜닝 시 `BATCH_SIZE=1` 또는 `2`부터 시작하는 것을 권장합니다.

In [ ]:
MODEL_NAME = "zhihan1996/DNABERT-2-117M"
DATA_PATH = None  # 예: Path("data/my_sequences.csv")
SEQUENCE_COLUMN = "sequence"
LABEL_COLUMN = "label"
MAX_LENGTH = 128
BATCH_SIZE = 4
EPOCHS = 3
FREEZE_ENCODER = True
LEARNING_RATE = 1e-3 if FREEZE_ENCODER else 2e-5
WEIGHT_DECAY = 0.01
OUTPUT_DIR = Path("outputs/dnabert2_classifier")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 2. DNABERT-2 로딩과 베이스 모델 출력 확인

공식 체크포인트에는 사용자 정의 모델 코드가 포함되어 있어 `trust_remote_code=True`가 필요합니다. 출처를 신뢰할 수 있는 체크포인트에서만 이 옵션을 사용하세요.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
config = BertConfig.from_pretrained(MODEL_NAME)
# 체크포인트의 구형 Triton 커널은 Windows의 최신 Triton과 호환되지 않습니다.
# attention dropout을 0보다 크게 설정하면 호환되는 PyTorch attention 경로가 사용됩니다.
config.attention_probs_dropout_prob = 0.1
encoder = AutoModel.from_pretrained(
    MODEL_NAME, trust_remote_code=True, config=config, add_pooling_layer=False
)
encoder.to(DEVICE)
encoder.eval()

example_sequences = [
    "ACGTACGTACGTACGT",
    "TTGCGTACCAAGTCTA",
]
encoded = tokenizer(
    example_sequences, padding=True, truncation=True,
    max_length=MAX_LENGTH, return_tensors="pt",
)
encoded_device = {k: v.to(DEVICE) for k, v in encoded.items()}
with torch.no_grad():
    base_outputs = encoder(**encoded_device)

token_embeddings = base_outputs[0]
attention_mask = encoded_device["attention_mask"]
mask = attention_mask.unsqueeze(-1).to(token_embeddings.dtype)
mean_embeddings = (token_embeddings * mask).sum(1) / mask.sum(1).clamp(min=1)

print("입력 토큰 shape       :", tuple(encoded["input_ids"].shape))
print("토큰 임베딩 shape     :", tuple(token_embeddings.shape))
print("서열 임베딩 shape     :", tuple(mean_embeddings.shape))
print("첫 8개 임베딩 값      :", mean_embeddings[0, :8].cpu().numpy())
print("임베딩 길이    :", len(mean_embeddings[0,]))


## 3. 라벨 데이터 불러오기 또는 생성

실제 CSV에는 DNA 서열과 정수 라벨이 있어야 합니다. 예시는 다음과 같습니다.

```csv
sequence,label
ACGTACGTACGT,0
TTGCAATAAAGC,1
```

유전체 실험에서는 무작위 분할보다 염색체 단위 분할이 권장됩니다. 파일에 `chromosome` 열이 있다면 아래 무작위 분할 코드를 특정 염색체를 test로 분리하는 방식으로 교체하여 서열 누출을 줄이세요.

In [ ]:
def make_synthetic_dataset(n_per_class=80, length=100, motif="TATAAA"):
    rng = np.random.default_rng(SEED)
    alphabet = np.array(list("ACGT"))
    rows = []
    for label in (0, 1):
        for _ in range(n_per_class):
            seq = "".join(rng.choice(alphabet, size=length))
            if label == 1:
                pos = int(rng.integers(0, length - len(motif) + 1))
                seq = seq[:pos] + motif + seq[pos + len(motif):]
            rows.append({SEQUENCE_COLUMN: seq, LABEL_COLUMN: label})
    return pd.DataFrame(rows).sample(frac=1, random_state=SEED).reset_index(drop=True)

if DATA_PATH is None:
    df = make_synthetic_dataset()
    print("합성 데이터를 사용합니다. 이 결과는 파이프라인 확인용입니다.")
else:
    df = pd.read_csv(DATA_PATH)
    print("불러온 파일:", DATA_PATH)

required = {SEQUENCE_COLUMN, LABEL_COLUMN}
missing = required.difference(df.columns)
if missing:
    raise ValueError(f"필수 열이 없습니다: {sorted(missing)}")
df = df.dropna(subset=[SEQUENCE_COLUMN, LABEL_COLUMN]).copy()
df[SEQUENCE_COLUMN] = df[SEQUENCE_COLUMN].astype(str).str.upper().str.replace(r"[^ACGTN]", "N", regex=True)
df[LABEL_COLUMN] = df[LABEL_COLUMN].astype(int)
if not set(df[LABEL_COLUMN].unique()).issubset({0, 1}):
    raise ValueError("이 노트북은 정수형 이진 라벨 0과 1을 사용합니다.")
print(df.head())
print("클래스별 샘플 수:\n", df[LABEL_COLUMN].value_counts().sort_index())


In [ ]:
train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df[LABEL_COLUMN], random_state=SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df[LABEL_COLUMN], random_state=SEED
)
print({"train": len(train_df), "validation": len(val_df), "test": len(test_df)})


## 4. Dataset과 분류 모델 정의

분류 모델은 padding을 제외한 토큰 임베딩을 평균 pooling한 다음 dropout과 선형 분류 레이어를 적용합니다.

In [ ]:
class DNASequenceDataset(Dataset):
    def __init__(self, frame):
        self.sequences = frame[SEQUENCE_COLUMN].tolist()
        self.labels = frame[LABEL_COLUMN].astype(int).tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]

def collate_batch(batch):
    sequences, labels = zip(*batch)
    tokens = tokenizer(
        list(sequences), padding=True, truncation=True,
        max_length=MAX_LENGTH, return_tensors="pt",
    )
    tokens["labels"] = torch.tensor(labels, dtype=torch.long)
    return tokens

train_loader = DataLoader(DNASequenceDataset(train_df), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_batch)
val_loader = DataLoader(DNASequenceDataset(val_df), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_batch)
test_loader = DataLoader(DNASequenceDataset(test_df), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_batch)

class DNABERT2Classifier(nn.Module):
    def __init__(self, encoder_model, num_labels=2, dropout=0.1):
        super().__init__()
        self.encoder = encoder_model
        hidden_size = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, num_labels)

    def forward(self, input_ids, attention_mask, **kwargs):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        hidden = outputs[0]
        mask = attention_mask.unsqueeze(-1).to(hidden.dtype)
        pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        return self.classifier(self.dropout(pooled))

model = DNABERT2Classifier(encoder).to(DEVICE)
if FREEZE_ENCODER:
    for parameter in model.encoder.parameters():
        parameter.requires_grad = False
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"학습 대상 파라미터: {trainable:,} / 전체 {total:,}")


## 5. 모델 학습과 최적 validation 체크포인트 선택

In [ ]:
def collect_predictions(model, loader):
    model.eval()
    all_labels, all_scores = [], []
    with torch.no_grad():
        for batch in loader:
            labels = batch.pop("labels")
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            logits = model(**batch)
            scores = torch.softmax(logits, dim=-1)[:, 1]
            all_labels.extend(labels.numpy().tolist())
            all_scores.extend(scores.cpu().numpy().tolist())
    return np.asarray(all_labels), np.asarray(all_scores)

def binary_metrics(labels, scores, threshold=0.5):
    predictions = (scores >= threshold).astype(int)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision_score(labels, predictions, zero_division=0),
        "recall": recall_score(labels, predictions, zero_division=0),
        "f1": f1_score(labels, predictions, zero_division=0),
        "mcc": matthews_corrcoef(labels, predictions),
        "auroc": roc_auc_score(labels, scores),
        "auprc": average_precision_score(labels, scores),
    }

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY,
)
criterion = nn.CrossEntropyLoss()
best_val_auprc = -np.inf
checkpoint_path = OUTPUT_DIR / "best_model.pt"
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    if FREEZE_ENCODER:
        model.encoder.eval()
    running_loss = 0.0
    for batch in train_loader:
        labels = batch.pop("labels").to(DEVICE)
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        optimizer.zero_grad(set_to_none=True)
        logits = model(**batch)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * labels.size(0)

    val_labels, val_scores = collect_predictions(model, val_loader)
    val_metrics = binary_metrics(val_labels, val_scores)
    row = {"epoch": epoch, "train_loss": running_loss / len(train_loader.dataset), **val_metrics}
    history.append(row)
    print(row)
    if val_metrics["auprc"] > best_val_auprc:
        best_val_auprc = val_metrics["auprc"]
        torch.save(model.state_dict(), checkpoint_path)

history_df = pd.DataFrame(history)
history_df.to_csv(OUTPUT_DIR / "training_history.csv", index=False)
history_df


## 6. 최종 test set 평가

validation AUPRC를 기준으로 최적 체크포인트를 선택한 다음에만 test set을 평가합니다. 논문 수준의 실험에서는 여러 random seed로 반복하고 평균 ± 표준편차를 보고하세요.

In [ ]:
model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE, weights_only=True))
test_labels, test_scores = collect_predictions(model, test_loader)
test_predictions = (test_scores >= 0.5).astype(int)
test_metrics = binary_metrics(test_labels, test_scores)

print("Test 성능 지표:")
for name, value in test_metrics.items():
    print(f"  {name:>9s}: {value:.4f}")
print("\n분류 성능 보고서:\n", classification_report(test_labels, test_predictions, digits=4, zero_division=0))
print("혼동 행렬:\n", confusion_matrix(test_labels, test_predictions))
pd.DataFrame([test_metrics]).to_csv(OUTPUT_DIR / "test_metrics.csv", index=False)


## 7. 새로운 DNA 서열 예측

아래 확률은 생물학적으로 유효한 라벨 데이터로 모델을 학습한 후에만 의미가 있습니다.

In [ ]:
def predict_sequences(sequences):
    model.eval()
    tokens = tokenizer(
        sequences, padding=True, truncation=True,
        max_length=MAX_LENGTH, return_tensors="pt",
    )
    tokens = {k: v.to(DEVICE) for k, v in tokens.items()}
    with torch.no_grad():
        probabilities = torch.softmax(model(**tokens), dim=-1)[:, 1].cpu().numpy()
    return pd.DataFrame({
        "sequence": sequences,
        "class_1_probability": probabilities,
        "predicted_label": (probabilities >= 0.5).astype(int),
    })

predict_sequences([
    "ACGTACGTACGTACGTACGT",
    "GGGCTATAAATTTCCCGGGC",
])

## 실제 데이터 적용 체크리스트

- 합성 데이터를 `sequence,label` 형식의 실제 CSV로 교체합니다.
- 데이터를 나누기 전에 중복 및 유사 중복 서열을 제거합니다.
- 염색체 또는 유전체 영역을 고려해 train/validation/test를 분할합니다.
- 토큰화된 길이 분포를 확인한 뒤 `MAX_LENGTH`를 정합니다. 잘린 구간에 중요한 신호가 있을 수 있습니다.
- encoder 동결 학습과 전체 파인튜닝 결과를 비교합니다.
- AUROC, AUPRC, MCC, F1 및 여러 seed의 변동성을 보고합니다.
- 모델과 임계값이 확정될 때까지 test set을 학습 및 설정 결정에 사용하지 않습니다.